In [1]:
import pandas as pd
import numpy as np
import ssl
import certifi
from urllib.request import urlopen
import json
from tqdm import tqdm
from datetime import datetime, date
import calendar
from urllib.error import HTTPError
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from functools import lru_cache
from DATA.stock_invest_function import *

In [3]:
# =============================================================================
# 최적화된 설정 및 유틸리티 함수들
# =============================================================================

# API 키 상수화
API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
BASE_URL = "https://financialmodelingprep.com/api/v3"

# 세션 생성 (연결 재사용)
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0'})

# 스레드 로컬 스토리지 (각 스레드마다 별도 세션)
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, 'session'):
        thread_local.session = requests.Session()
        thread_local.session.headers.update({'User-Agent': 'Mozilla/5.0'})
    return thread_local.session

@lru_cache(maxsize=128)
def get_last_quarter_end():
    today = date.today()
    quarter_months = [3, 6, 9, 12]

    for m in quarter_months:
        if today.month <= m:
            last_quarter_month = quarter_months[quarter_months.index(m) - 1]
            break
    else:
        last_quarter_month = 12

    year = today.year if last_quarter_month != 12 else today.year - 1
    last_day = calendar.monthrange(year, last_quarter_month)[1]
    return date(year, last_quarter_month, last_day)

@lru_cache(maxsize=128)
def get_last_month_end():
    today = date.today()
    last_month = today.month - 1 if today.month > 1 else 12
    last_year = today.year if today.month > 1 else today.year - 1
    last_day = calendar.monthrange(last_year, last_month)[1]
    last_month_end = date(last_year, last_month, last_day)
    return last_month_end.strftime("%Y-%m-%d")

# 최적화된 데이터 요청 함수 (requests 사용)
def get_data_with_retry(url, max_retries=3, delay=0.1):
    """재시도 로직이 포함된 데이터 요청 함수"""
    session = get_session()

    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=30)

            if response.status_code == 429:  # Rate limit
                sleep_time = min(60 * (2 ** attempt), 300)  # 지수 백오프, 최대 5분
                print(f"Rate limit hit. Sleeping for {sleep_time} seconds...")
                time.sleep(sleep_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"Failed to fetch {url}: {e}")
                return None
            time.sleep(delay * (2 ** attempt))  # 지수 백오프

    return None

# =============================================================================
# 1. 주식 리스트 및 프로필 데이터 수집 (최적화)
# =============================================================================

print("Step 1: Fetching stock list...")

# 주식 리스트 가져오기
stock_list_url = f"{BASE_URL}/stock/list?apikey={API_KEY}"
stock_list = get_data_with_retry(stock_list_url)

if not stock_list:
    raise Exception("Failed to fetch stock list")

# DataFrame 변환 및 필터링 (벡터화 연산 사용)
stock_listed = pd.DataFrame(stock_list)

# 조건을 한 번에 처리
target_exchanges = ['NASDAQ', 'NYSE', 'AMEX']
us_stock_info = stock_listed[
    (stock_listed['exchangeShortName'].isin(target_exchanges)) &
    (stock_listed['type'] == 'stock')
].copy()

ticker_list = us_stock_info['symbol'].unique().tolist()
print(f"Found {len(ticker_list)} tickers")

# =============================================================================
# 2. 병렬 처리로 프로필 데이터 수집
# =============================================================================

def fetch_profile(ticker):
    """단일 티커의 프로필 데이터를 가져오는 함수"""
    url = f"{BASE_URL}/profile/{ticker}?apikey={API_KEY}"
    data = get_data_with_retry(url)

    if data and isinstance(data, list) and len(data) > 0:
        return pd.DataFrame(data)
    return None

print("Step 2: Fetching stock profiles with parallel processing...")

info_list = []
MAX_WORKERS = 10  # 동시 요청 수 제한

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 배치 처리로 메모리 효율성 향상
    batch_size = 100

    for i in range(0, len(ticker_list), batch_size):
        batch = ticker_list[i:i+batch_size]

        # 병렬 요청 제출
        future_to_ticker = {executor.submit(fetch_profile, ticker): ticker
                           for ticker in batch}

        # 결과 수집
        for future in tqdm(as_completed(future_to_ticker),
                          total=len(batch),
                          desc=f"Batch {i//batch_size + 1}"):
            result = future.result()
            if result is not None:
                info_list.append(result)

        # 배치 간 짧은 딜레이
        if i + batch_size < len(ticker_list):
            time.sleep(1)

# 데이터 병합
if info_list:
    info = pd.concat(info_list, ignore_index=True)
else:
    info = pd.DataFrame()
    print("No profile data collected")

# =============================================================================
# 3. 데이터 필터링 및 티커 리스트 생성
# =============================================================================

print("Step 3: Filtering data...")

# 날짜 정보
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
last_quarter_end = get_last_quarter_end()
quarter_date = last_quarter_end.strftime("%Y-%m-%d")
last_month_date = get_last_month_end()

print(f"Last quarter end: {quarter_date}")
print(f"Last month end: {last_month_date}")

# 섹터 필터링 (벡터화 연산)
if not info.empty:
    info_df = info[['symbol', 'sector', 'industry']].copy()

    excluded_sectors = ['Financial Services', 'Real Estate', 'Utilities', '']
    info_re = info_df[~info_df['sector'].isin(excluded_sectors)].copy()
    tics_list = info_re['symbol'].unique().tolist()

    print(f"Filtered to {len(tics_list)} tickers")

    # 티커 리스트 저장 - DataFrame으로 저장
    ticker_df = pd.DataFrame(tics_list, columns=['ticker'])
    # pd.DataFrame(tics_list, columns=['ticker']).to_csv(
    #     f'/content/drive/MyDrive/Stock_Investment_Strategy/Factor_Model/Data/US_stock_prc/ticker_list_{today_date}.csv',
    #     index=False
    # )
else:
    # 기존 파일에서 읽기
    path = '/content/drive/MyDrive/Stock_Investment_Strategy/Factor_Model/Data/US_stock_prc/ticker_list_2025-05-28.csv'
    ticker_data = pd.read_csv(path)
    tics_list = ticker_data['ticker'].unique().tolist()

# =============================================================================
# 4. 재무제표 데이터 수집 (병렬 처리)
# =============================================================================

# 컬럼 매핑 정의
COLUMN_MAPPINGS = {
    'income_statement': {
        'compustat_cols': ['report_date', 'ticker', 'period', 'sale', 'cogs', 'gp', 'xrd', 'xsga',
                          'idit', 'xint', 'dp', 'ebitda', 'xopr', 'opiti', 'opir', 'pi', 'pir',
                          'txt', 'ni', 'nir', 'eps', 'epsdi', 'shrout', 'shroutdi'],
        'fmp_cols': ['date', 'symbol','period', 'revenue', 'costOfRevenue', 'grossProfit',
                    'researchAndDevelopmentExpenses', 'sellingGeneralAndAdministrativeExpenses',
                    'interestIncome', 'interestExpense', 'depreciationAndAmortization', 'ebitda',
                    'operatingExpenses', 'operatingIncome', 'operatingIncomeRatio', 'incomeBeforeTax',
                    'incomeBeforeTaxRatio','incomeTaxExpense', 'netIncome', 'netIncomeRatio', 'eps',
                    'epsdiluted', 'weightedAverageShsOut','weightedAverageShsOutDil'],
        'endpoint': 'income-statement'
    },
    'balance_sheet': {
        'compustat_cols': ['report_date', 'ticker', 'at', 'ca', 'rec', 'cash', 'invt', 'intan',
                          'ivao', 'ppen', 'ao', 'lt', 'lo', 'debtst', 'ap', 'txp', 'debtlt',
                          'pstk', 'be', 'debt', 'netdebt'],
        'fmp_cols': ['date', 'symbol', 'totalAssets', 'totalCurrentAssets', 'netReceivables',
                    'cashAndCashEquivalents', 'inventory', 'intangibleAssets', 'investments',
                    'propertyPlantEquipmentNet', 'otherAssets', 'totalLiabilities', 'otherLiabilities',
                    'shortTermDebt', 'accountPayables', 'taxPayables', 'longTermDebt',
                    'preferredStock', 'totalStockholdersEquity', 'totalDebt', 'netDebt'],
        'endpoint': 'balance-sheet-statement'
    },
    'cash_flow': {
        'compustat_cols': ['report_date', 'ticker', 'capx', 'ocf', 'eqbb', 'eqis', 'dstnetis',
                          'dltnetis', 'fincf', 'fcf'],
        'fmp_cols': ['date', 'symbol', 'capitalExpenditure', 'operatingCashFlow',
                    'commonStockRepurchased', 'commonStockIssued', 'debtRepayment',
                    'otherFinancingActivites', 'netCashUsedProvidedByFinancingActivities',
                    'freeCashFlow'],
        'endpoint': 'cash-flow-statement'
    }
}

def process_financial_statement(ticker, statement_type, dates_list):
    """재무제표 데이터를 처리하는 함수"""
    mapping = COLUMN_MAPPINGS[statement_type]

    try:
        url = f"{BASE_URL}/{mapping['endpoint']}/{ticker}?period=quarter&apikey={API_KEY}"
        fs_raw = get_data_with_retry(url)

        if not fs_raw or not isinstance(fs_raw, list):
            return None

        temp_df = pd.DataFrame(fs_raw)

        # 필수 컬럼 확인
        required_cols = set(mapping['fmp_cols'])
        available_cols = set(temp_df.columns)

        # 사용 가능한 컬럼만 선택
        cols_to_use = [col for col in mapping['fmp_cols'] if col in available_cols]

        if 'date' not in cols_to_use or 'symbol' not in cols_to_use:
            return None

        fs_df = temp_df[cols_to_use].copy()

        # 컬럼 이름 매핑 (사용 가능한 것만)
        col_mapping = dict(zip(cols_to_use, mapping['compustat_cols'][:len(cols_to_use)]))
        fs_df.columns = [col_mapping.get(col, col) for col in fs_df.columns]

        # 날짜 처리 및 정렬
        fs_df['report_date'] = pd.to_datetime(fs_df['report_date'])
        fs_df_sorted = fs_df.sort_values(by='report_date')

        # 월별 데이터로 변환
        fs_df_sorted['date_month'] = fs_df_sorted['report_date'].dt.to_period('M').astype(str)
        fs_df_sorted['date'] = pd.to_datetime(fs_df_sorted['date_month']) + pd.offsets.MonthEnd(0)

        # 기준 날짜와 병합
        date_df = pd.DataFrame(dates_list, columns=['date'])
        temp_fs = pd.merge(date_df, fs_df_sorted, on=['date'], how='left').ffill()

        return temp_fs

    except Exception as e:
        print(f"Error processing {statement_type} for {ticker}: {e}")
        return None

def collect_financial_data(statement_type, tics_list, dates_list):
    """병렬로 재무제표 데이터 수집"""
    print(f"Step 4.{['income_statement', 'balance_sheet', 'cash_flow'].index(statement_type)+1}: Collecting {statement_type} data...")

    fs_list = []
    error_list = []
    MAX_WORKERS = 8  # 병렬 처리 워커 수

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 배치 처리
        batch_size = 50

        for i in range(0, len(tics_list), batch_size):
            batch = tics_list[i:i+batch_size]

            # 병렬 작업 제출
            future_to_ticker = {
                executor.submit(process_financial_statement, ticker, statement_type, dates_list): ticker
                for ticker in batch
            }

            # 결과 수집
            for future in tqdm(as_completed(future_to_ticker),
                             total=len(batch),
                             desc=f"{statement_type} batch {i//batch_size + 1}"):
                ticker = future_to_ticker[future]
                try:
                    result = future.result()
                    if result is not None:
                        fs_list.append(result)
                    else:
                        error_list.append(ticker)
                except Exception as e:
                    print(f"Error for {ticker}: {e}")
                    error_list.append(ticker)

            # 배치 간 딜레이
            if i + batch_size < len(tics_list):
                time.sleep(0.5)

    # 데이터 병합
    if fs_list:
        fs_df = pd.concat(fs_list, ignore_index=True)
        fs_df = fs_df[fs_df['ticker'].notna()]
    else:
        fs_df = pd.DataFrame()

    print(f"{statement_type} - Collected: {len(fs_list)}, Errors: {len(error_list)}")
    return fs_df, error_list

# =============================================================================
# 5. 데이터 수집 실행
# =============================================================================

# 날짜 범위 설정
dates_list = pd.date_range('2004-01-31', last_month_date, freq='M')

# Income Statement 수집
is_df, is_errors = collect_financial_data('income_statement', tics_list, dates_list)
# if not is_df.empty:
#     is_df.to_csv(f'/content/drive/MyDrive/Stock_Investment_Strategy/Factor_Model/Data/US_IS/us_is_{today_date}.csv', index=False)

# Balance Sheet 수집
bs_df, bs_errors = collect_financial_data('balance_sheet', tics_list, dates_list)
# if not bs_df.empty:
#     bs_df.to_csv(f'/content/drive/MyDrive/Stock_Investment_Strategy/Factor_Model/Data/US_BS/us_bs_{today_date}.csv', index=False)

# Cash Flow 수집
cf_df, cf_errors = collect_financial_data('cash_flow', tics_list, dates_list)
# if not cf_df.empty:
#     cf_df.to_csv(f'/content/drive/MyDrive/Stock_Investment_Strategy/Factor_Model/Data/US_CF/us_cf_{today_date}.csv', index=False)

# =============================================================================
# 6. 결과 요약
# =============================================================================

print("\n=== Data Collection Summary ===")
print(f"Total tickers processed: {len(tics_list)}")
print(f"Income Statement - Success: {len(is_df)//len(dates_list) if not is_df.empty else 0}, Errors: {len(is_errors)}")
print(f"Balance Sheet - Success: {len(bs_df)//len(dates_list) if not bs_df.empty else 0}, Errors: {len(bs_errors)}")
print(f"Cash Flow - Success: {len(cf_df)//len(dates_list) if not cf_df.empty else 0}, Errors: {len(cf_errors)}")
# print(f"Files saved to respective directories with date: {today_date}")
print(f"DataFrames created:")
print(f"  - ticker_df: {len(ticker_df)} tickers")
print(f"  - is_df: {is_df.shape if not is_df.empty else 'Empty'}")
print(f"  - bs_df: {bs_df.shape if not bs_df.empty else 'Empty'}")
print(f"  - cf_df: {cf_df.shape if not cf_df.empty else 'Empty'}")

# 세션 정리
session.close()
print("Data collection completed!")

# =============================================================================
# 7. 수집된 DataFrame들 확인
# =============================================================================

print("\n=== Available DataFrames ===")
print("1. ticker_df - Filtered ticker list")
print("2. is_df - Income Statement data")
print("3. bs_df - Balance Sheet data")
print("4. cf_df - Cash Flow data")
print("5. info - Company profile data")
print("6. us_stock_info - US stock information")

income_statement batch 95:  76%|███████▌  | 38/50 [00:01<00:00, 32.51it/s]

Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...


income_statement batch 104:   0%|          | 0/50 [00:00<?, ?it/s]

Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...
Rate limit hit. Sleeping for 60 seconds...


income_statement batch 110:   0%|          | 0/50 [00:00<?, ?it/s]

...

Rate limit hit. Sleeping for 60 seconds...


balance_sheet batch 35:  96%|█████████▌| 48/50 [00:02<00:00, 28.43it/s]

Rate limit hit. Sleeping for 60 seconds...


balance_sheet batch 35:  98%|█████████▊| 49/50 [01:03<00:01,  1.29s/it]


KeyboardInterrupt: 

In [31]:
# =============================================================================
# CONSTANTS AND CONFIGURATION
# =============================================================================

import gc

# Balance Sheet 컬럼 매핑
BALANCE_SHEET_ITEMS = [
    'at', 'ca', 'rec', 'cash', 'invt', 'intan', 'ivao', 'ppen', 'ao',
    'lt', 'lo', 'debtst', 'ap', 'txp', 'debtlt', 'pstk', 'be', 'debt', 'netdebt'
]

# Cash Flow 컬럼 매핑
CASH_FLOW_ITEMS = [
    'capx', 'ocf', 'eqbb', 'eqis', 'dstnetis', 'dltnetis', 'fincf', 'fcf'
]

# Income Statement 컬럼
INCOME_STATEMENT_ITEMS = [
    'sale', 'cogs', 'gp', 'xrd', 'xsga', 'idit', 'xint', 'dp', 'ebitda',
    'xopr', 'opiti', 'opir', 'pi', 'pir', 'txt', 'ni', 'nir', 'eps',
    'epsdi', 'shrout', 'shroutdi'
]

# =============================================================================
# UTILITY FUNCTIONS - DEPRECATED 함수들 수정
# =============================================================================

def is_categorical_column(series_or_dtype):
    """
    Series나 dtype이 categorical인지 확인하는 함수
    pandas deprecated 함수 대신 사용
    """
    if hasattr(series_or_dtype, 'dtype'):
        return isinstance(series_or_dtype.dtype, pd.CategoricalDtype)
    else:
        return isinstance(series_or_dtype, pd.CategoricalDtype)

def analyze_dataframe_structure(df, df_name="DataFrame"):
    """
    DataFrame의 구조를 분석하고 value 컬럼을 식별하는 함수
    """
    print(f"\n{'='*50}")
    print(f"ANALYZING {df_name.upper()} STRUCTURE")
    print(f"{'='*50}")

    # 기본 정보
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # 식별 컬럼들
    id_columns = ['date', 'report_date', 'ticker', 'period', 'date_month']
    available_id_cols = [col for col in id_columns if col in df.columns]
    print(f"Available ID columns: {available_id_cols}")

    # 기본 financial items
    default_financial_items = [
        'sale', 'cogs', 'gp', 'xrd', 'xsga', 'idit', 'xint', 'dp', 'ebitda',
        'xopr', 'opiti', 'opir', 'pi', 'pir', 'txt', 'ni', 'nir', 'eps',
        'epsdi', 'shrout', 'shroutdi'
    ]

    available_default_items = [col for col in default_financial_items if col in df.columns]
    print(f"Available default financial items ({len(available_default_items)}): {available_default_items}")

    # 추가 컬럼들 (ID도 아니고 기본 financial item도 아닌 것들)
    other_columns = [col for col in df.columns
                    if col not in available_id_cols and col not in available_default_items]

    if other_columns:
        print(f"Additional columns ({len(other_columns)}): {other_columns}")

        # 숫자형 컬럼인지 확인
        numeric_others = []
        non_numeric_others = []

        for col in other_columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_others.append(col)
            else:
                non_numeric_others.append(col)

        if numeric_others:
            print(f"  - Numeric (potential financial items): {numeric_others}")
        if non_numeric_others:
            print(f"  - Non-numeric: {non_numeric_others}")

        return available_default_items + numeric_others

    return available_default_items

def add_fs_name_to_dataframes(is_df=None, bs_df=None, cf_df=None):
    """기존 DataFrame들에 fs_name 컬럼을 추가하는 함수"""

    results = {}

    # Income Statement DataFrame에 fs_name 추가
    if is_df is not None and not is_df.empty:
        if 'fs_name' not in is_df.columns:
            is_df = is_df.copy()
            is_df['fs_name'] = 'is'
            print("Added 'fs_name' = 'is' to is_df")
        results['is_df'] = is_df

    # Balance Sheet DataFrame에 fs_name 추가
    if bs_df is not None and not bs_df.empty:
        if 'fs_name' not in bs_df.columns:
            bs_df = bs_df.copy()
            bs_df['fs_name'] = 'bs'
            print("Added 'fs_name' = 'bs' to bs_df")
        results['bs_df'] = bs_df

    # Cash Flow DataFrame에 fs_name 추가
    if cf_df is not None and not cf_df.empty:
        if 'fs_name' not in cf_df.columns:
            cf_df = cf_df.copy()
            cf_df['fs_name'] = 'cf'
            print("Added 'fs_name' = 'cf' to cf_df")
        results['cf_df'] = cf_df

    return results

def validate_long_format_data(df):
    """Long format 데이터의 품질을 검증"""

    print("\n" + "="*50)
    print("DATA QUALITY VALIDATION")
    print("="*50)

    # 기본 정보
    print(f"Total records: {len(df):,}")
    if 'ticker' in df.columns:
        print(f"Unique tickers: {df['ticker'].nunique():,}")
    if 'item' in df.columns:
        print(f"Unique items: {df['item'].nunique():,}")

    # 결측값 체크
    for col in ['ticker', 'date', 'item', 'value']:
        if col in df.columns:
            miss = df[col].isna().sum()
            frac = (miss / len(df) * 100) if len(df) else 0
            print(f"- {col}: {miss:,} missing ({frac:.1f}%)")

    # 중복값 체크
    key = [c for c in ['ticker', 'date', 'item'] if c in df.columns]
    if key:
        duplicates = df.duplicated(subset=key).sum()
        print(f"\nDuplicate records ({'-'.join(key)}): {duplicates:,}")

    # 값 분포
    if 'value' in df.columns:
        print(f"\nValue statistics:")
        print(df['value'].describe())

    # 티커별 레코드 수
    if 'ticker' in df.columns:
        ticker_counts = df['ticker'].value_counts()
        print(f"\nRecords per ticker - Min: {ticker_counts.min()}, Max: {ticker_counts.max()}, Mean: {ticker_counts.mean():.0f}")

    return df

# =============================================================================
# CORE CONVERSION FUNCTIONS (STREAMING / CHUNKED) - DEPRECATED 함수들 수정
# =============================================================================

def _prepare_ids_and_values(df, custom_value_vars):
    """id_vars / value_vars 자동 추출 + 보정"""
    # id 후보
    id_vars_all = ['date', 'report_date', 'ticker', 'period', 'date_month']
    id_vars = [c for c in id_vars_all if c in df.columns]

    # 기본 재무항목
    default_financial_items = [
        'sale','cogs','gp','xrd','xsga','idit','xint','dp','ebitda',
        'xopr','opiti','opir','pi','pir','txt','ni','nir','eps',
        'epsdi','shrout','shroutdi'
    ]
    if custom_value_vars is not None:
        base_vars = [c for c in custom_value_vars if c in df.columns]
        print(f"Using custom value variables: {len(base_vars)} columns")
    else:
        base_vars = [c for c in default_financial_items if c in df.columns]

    # 기본 리스트 외 컬럼도 자동 포함
    other_cols = [c for c in df.columns if c not in id_vars and c not in base_vars]
    if other_cols:
        print(f"Additional financial columns detected: {other_cols}")
        base_vars.extend(other_cols)

    value_vars = base_vars

    print(f"ID variables: {id_vars}")
    print(f"Value variables: {value_vars}")
    print(f"Total financial items: {len(value_vars)}")
    return id_vars, value_vars

def _optimize_dtypes_inplace(df):
    """ticker/period을 category로, 날짜를 datetime으로, 숫자 다운캐스트 - deprecated 함수 수정"""
    # ticker를 카테고리로 변환 (수정된 방식)
    if 'ticker' in df.columns and not is_categorical_column(df['ticker']):
        df['ticker'] = df['ticker'].astype('category')

    # period을 카테고리로 변환
    if 'period' in df.columns and df['period'].nunique() < 64:
        df['period'] = df['period'].astype('category')

    # 날짜 컬럼 처리
    for dcol in ('date', 'report_date'):
        if dcol in df.columns and not pd.api.types.is_datetime64_any_dtype(df[dcol]):
            df[dcol] = pd.to_datetime(df[dcol], errors='coerce')

def convert_wide_to_long(
    df,
    custom_value_vars=None,
    fs_name=None,
    col_chunk_size=16,      # 재무항목 수 청크
    row_chunk_size=100_000, # 행 청크
    do_sort=False
):
    """
    메모리 안전 버전 - deprecated 함수들 수정:
      - (1) value_vars를 col_chunk_size로 나눠서
      - (2) 행도 row_chunk_size로 나눠서
      - 순차 melt → concat
    """
    import math

    # 사전 최적화
    df_local = df.copy()
    _optimize_dtypes_inplace(df_local)

    # id / value 설정
    id_vars, value_vars = _prepare_ids_and_values(df_local, custom_value_vars)

    # 결과 조각 모으기
    out_chunks = []
    n_rows = len(df_local)
    n_row_chunks = math.ceil(n_rows / row_chunk_size)

    # 행-청크 루프
    for r in range(n_row_chunks):
        r0 = r * row_chunk_size
        r1 = min(n_rows, (r+1) * row_chunk_size)
        part = df_local.iloc[r0:r1, :].copy()

        # 열-청크 루프
        for c in range(0, len(value_vars), col_chunk_size):
            vv = value_vars[c:c+col_chunk_size]

            # melt (작은 조각)
            melted = part.melt(
                id_vars=id_vars,
                value_vars=vv,
                var_name='item',
                value_name='value'
            )

            # 숫자 다운캐스트
            melted['value'] = pd.to_numeric(melted['value'], errors='coerce', downcast='float')

            # fs_name
            if fs_name:
                melted['fs_name'] = fs_name

            # 카테고리화 (수정된 방식)
            if not is_categorical_column(melted['item']):
                melted['item'] = melted['item'].astype('category')
            if 'fs_name' in melted.columns and not is_categorical_column(melted['fs_name']):
                melted['fs_name'] = melted['fs_name'].astype('category')

            out_chunks.append(melted)

            # 메모리 회수
            del melted
            gc.collect()

        del part
        gc.collect()

    # 최종 결합
    if out_chunks:
        long_df = pd.concat(out_chunks, ignore_index=True)
        del out_chunks
        gc.collect()
    else:
        long_df = pd.DataFrame(columns=id_vars + ['item', 'value'] + (['fs_name'] if fs_name else []))

    # (선택) 정렬 — 대규모 데이터에서 메모리 피크 커지므로 기본 off
    if do_sort:
        sort_cols = [c for c in ['ticker', 'date', 'item'] if c in long_df.columns]
        if sort_cols:
            long_df = long_df.sort_values(sort_cols, kind='quicksort').reset_index(drop=True)

    return long_df

def convert_financial_data_to_long(df, statement_type="auto", fs_name=None,
                                   col_chunk_size=16, row_chunk_size=100_000,
                                   do_sort=False):
    """
    재무제표 유형에 따라 적절한 컬럼을 선택하여 long format으로 변환 (청크형)
    """
    if statement_type == "auto":
        print("Auto-detecting statement type...")
        is_matches = len([c for c in INCOME_STATEMENT_ITEMS if c in df.columns])
        bs_matches = len([c for c in BALANCE_SHEET_ITEMS if c in df.columns])
        cf_matches = len([c for c in CASH_FLOW_ITEMS if c in df.columns])
        print(f"Column matches - IS: {is_matches}, BS: {bs_matches}, CF: {cf_matches}")

        if is_matches >= bs_matches and is_matches >= cf_matches:
            statement_type = "income"; target_items = INCOME_STATEMENT_ITEMS; fs_name = fs_name or "is"
        elif bs_matches >= cf_matches:
            statement_type = "balance"; target_items = BALANCE_SHEET_ITEMS; fs_name = fs_name or "bs"
        else:
            statement_type = "cash_flow"; target_items = CASH_FLOW_ITEMS; fs_name = fs_name or "cf"
        print(f"Detected statement type: {statement_type} (fs_name: {fs_name})")
    elif statement_type == "income":
        target_items = INCOME_STATEMENT_ITEMS; fs_name = fs_name or "is"
    elif statement_type == "balance":
        target_items = BALANCE_SHEET_ITEMS; fs_name = fs_name or "bs"
    elif statement_type == "cash_flow":
        target_items = CASH_FLOW_ITEMS; fs_name = fs_name or "cf"
    else:
        raise ValueError("Invalid statement_type. Use 'income', 'balance', 'cash_flow', or 'auto'")

    return convert_wide_to_long(
        df,
        custom_value_vars=target_items,
        fs_name=fs_name,
        col_chunk_size=col_chunk_size,
        row_chunk_size=row_chunk_size,
        do_sort=do_sort,
    )

# =============================================================================
# MEMORY OPTIMIZED PROCESSING FUNCTION
# =============================================================================

def process_financial_data_individually(is_df=None, bs_df=None, cf_df=None):
    """
    메모리 최적화 버전 - 각 재무제표를 개별적으로만 long format으로 변환

    Parameters:
    -----------
    is_df : DataFrame, optional
        Income Statement 데이터프레임
    bs_df : DataFrame, optional
        Balance Sheet 데이터프레임
    cf_df : DataFrame, optional
        Cash Flow 데이터프레임

    Returns:
    --------
    dict : {'IS': is_long, 'BS': bs_long, 'CF': cf_long}
        각 재무제표의 long format 데이터프레임들을 담은 딕셔너리
    """

    # 파라미터가 없으면 전역 변수에서 찾기
    if is_df is None:
        is_df = globals().get('is_df', pd.DataFrame())
    if bs_df is None:
        bs_df = globals().get('bs_df', pd.DataFrame())
    if cf_df is None:
        cf_df = globals().get('cf_df', pd.DataFrame())

    # 결과를 저장할 딕셔너리
    results = {}

    print("="*70)
    print("MEMORY OPTIMIZED PROCESSING - INDIVIDUAL CONVERSION ONLY")
    print("="*70)

    # Income Statement 개별 처리
    if not is_df.empty:
        print("\n1. Processing Income Statement...")
        print("-" * 40)

        _ = analyze_dataframe_structure(is_df, "Income Statement")

        is_long = convert_financial_data_to_long(
            is_df.copy(),
            statement_type="income",
            fs_name="is",
            col_chunk_size=8,      # 더 작은 청크로 안전하게
            row_chunk_size=50_000,  # 행 청크도 줄임
            do_sort=False
        )

        print(f"✓ IS Long format completed: {is_long.shape}")
        results['IS'] = is_long

        # 검증
        print("\n[Income Statement Long Format Validation]")
        validate_long_format_data(is_long)

        # 메모리 정리
        del is_long
        gc.collect()
    else:
        print("\n1. Income Statement: No data provided")

    # Balance Sheet 개별 처리
    if not bs_df.empty:
        print("\n2. Processing Balance Sheet...")
        print("-" * 40)

        _ = analyze_dataframe_structure(bs_df, "Balance Sheet")

        bs_long = convert_financial_data_to_long(
            bs_df.copy(),
            statement_type="balance",
            fs_name="bs",
            col_chunk_size=8,
            row_chunk_size=50_000,
            do_sort=False
        )

        print(f"✓ BS Long format completed: {bs_long.shape}")
        results['BS'] = bs_long

        # 검증
        print("\n[Balance Sheet Long Format Validation]")
        validate_long_format_data(bs_long)

        # 메모리 정리
        del bs_long
        gc.collect()
    else:
        print("\n2. Balance Sheet: No data provided")

    # Cash Flow 개별 처리
    if not cf_df.empty:
        print("\n3. Processing Cash Flow...")
        print("-" * 40)

        _ = analyze_dataframe_structure(cf_df, "Cash Flow")

        cf_long = convert_financial_data_to_long(
            cf_df.copy(),
            statement_type="cash_flow",
            fs_name="cf",
            col_chunk_size=8,
            row_chunk_size=50_000,
            do_sort=False
        )

        print(f"✓ CF Long format completed: {cf_long.shape}")
        results['CF'] = cf_long

        # 검증
        print("\n[Cash Flow Long Format Validation]")
        validate_long_format_data(cf_long)

        # 메모리 정리
        del cf_long
        gc.collect()
    else:
        print("\n3. Cash Flow: No data provided")

    print("\n" + "="*70)
    print("INDIVIDUAL PROCESSING COMPLETED!")
    print("="*70)
    print(f"Created datasets: {list(results.keys())}")
    print("\n✓ Memory optimized - No combined dataset created")
    print("✓ Each dataset processed independently")
    print("✓ Reduced memory footprint significantly")
    print("✓ Fixed deprecated function warnings")

    return results

In [32]:
# 결과 사용 예제:
results = process_financial_data_individually()

# 개별 변수로 할당
IS = results.get('IS')  # Income Statement long format
BS = results.get('BS')  # Balance Sheet long format
CF = results.get('CF')  # Cash Flow long format

# 존재 여부 확인
if IS is not None:
    print(f"Income Statement: {IS.shape}")
    # IS 데이터 사용...

if BS is not None:
    print(f"Balance Sheet: {BS.shape}")
    # BS 데이터 사용...

if CF is not None:
    print(f"Cash Flow: {CF.shape}")
    # CF 데이터 사용...


MEMORY OPTIMIZED PROCESSING - INDIVIDUAL CONVERSION ONLY

1. Processing Income Statement...
----------------------------------------

ANALYZING INCOME STATEMENT STRUCTURE
Shape: (975965, 26)
Columns: ['date', 'report_date', 'ticker', 'period', 'sale', 'cogs', 'gp', 'xrd', 'xsga', 'idit', 'xint', 'dp', 'ebitda', 'xopr', 'opiti', 'opir', 'pi', 'pir', 'txt', 'ni', 'nir', 'eps', 'epsdi', 'shrout', 'shroutdi', 'date_month']
Available ID columns: ['date', 'report_date', 'ticker', 'period', 'date_month']
Available default financial items (21): ['sale', 'cogs', 'gp', 'xrd', 'xsga', 'idit', 'xint', 'dp', 'ebitda', 'xopr', 'opiti', 'opir', 'pi', 'pir', 'txt', 'ni', 'nir', 'eps', 'epsdi', 'shrout', 'shroutdi']
Using custom value variables: 21 columns
ID variables: ['date', 'report_date', 'ticker', 'period', 'date_month']
Value variables: ['sale', 'cogs', 'gp', 'xrd', 'xsga', 'idit', 'xint', 'dp', 'ebitda', 'xopr', 'opiti', 'opir', 'pi', 'pir', 'txt', 'ni', 'nir', 'eps', 'epsdi', 'shrout', 'shrout

In [39]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}


In [ ]:
from __future__ import annotations

import warnings
import time
import gc

from sqlalchemy import create_engine, text, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert
from tqdm import tqdm

warnings.filterwarnings('ignore')

connection_string = (
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
    f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
)

# 연결 풀 설정으로 안정성 향상
engine = create_engine(
    connection_string,
    echo=False,
    pool_size=5,
    max_overflow=10,
    pool_timeout=30,
    pool_recycle=3600,   # 1시간마다 연결 새로고침
    pool_pre_ping=True   # 연결 상태 미리 확인
)


# =========================================
# 유틸: UNIQUE 인덱스 보장
# =========================================
def _ensure_unique_index(conn, table_name: str, dedup_keys: list[str]):
    """
    dedup_keys 기준으로 UNIQUE INDEX가 없으면 생성
    """
    if not dedup_keys:
        return
    idx_name = f"uniq_{table_name}_" + "_".join(dedup_keys)
    cols = ", ".join([f"`{c}`" for c in dedup_keys])

    # 존재 여부 확인
    chk = conn.execute(
        text(f"SHOW INDEX FROM `{table_name}` WHERE Key_name = :k"),
        {"k": idx_name}
    ).fetchone()

    if not chk:
        conn.execute(text(f"ALTER TABLE `{table_name}` ADD UNIQUE INDEX `{idx_name}` ({cols})"))
        conn.commit()


# =========================================
# 유틸: INSERT IGNORE (중복은 무시)
# =========================================
def _insert_ignore_chunk(conn, table_name: str, chunk_df: pd.DataFrame) -> int:
    """
    MySQL INSERT IGNORE (중복키 충돌 시 무시)로 청크 삽입
    - 정확한 동작을 위해 테이블에 UNIQUE(dedup_keys) 인덱스가 있어야 함
    - return: 실제로 삽입된 행 수
    """
    if chunk_df.empty:
        return 0

    # 테이블 메타 반영 (Reflection)
    md = MetaData()
    tbl = Table(table_name, md, autoload_with=conn)

    rows = chunk_df.to_dict(orient='records')
    # INSERT IGNORE
    stmt = mysql_insert(tbl).values(rows).prefix_with("IGNORE")
    res = conn.execute(stmt)
    conn.commit()

    # res.rowcount는 실제로 "삽입된" 행 수를 반환(중복 무시는 카운트 X)
    return res.rowcount or 0


# =========================================
# 대용량 안전 저장 (중복 무시 모드)
# =========================================
def save_large_dataframe_safely(
    df: pd.DataFrame,
    table_name: str,
    engine,
    chunk_size: int = 5000,
    show_progress: bool = True,
    dedup_keys: list[str] | None = None,
    drop_table_first: bool = False,   # 기존 데이터 유지가 기본
    skip_existing: bool = True        # True면 기존 중복키 무시(INSERT IGNORE)
) -> bool:
    """
    대용량 DataFrame을 안전하게 MySQL에 저장

    - drop_table_first=False : 기존 테이블 유지 (없으면 생성)
    - skip_existing=True     : UNIQUE 인덱스 + INSERT IGNORE로 과거 중복은 "무시"(추가 적재 안 함)
    - dedup_keys             : UNIQUE 인덱스 생성용 키 목록 (예: ['ticker','date','item'])
    """
    if df is None or df.empty:
        print(f"   ❌ No data to save for {table_name}")
        return False

    total_rows = len(df)
    total_chunks = (total_rows + chunk_size - 1) // chunk_size
    print(f"   📊 Processing {total_rows:,} rows in chunks of {chunk_size:,} (total {total_chunks} chunks)")

    # 테이블 준비 (없으면 생성 / 있으면 유지)
    # 첫 청크를 임시로 사용해 스키마를 잡을 수 있음
    first_chunk = df.iloc[0: min(chunk_size, total_rows)].copy()

    try:
        with engine.connect() as conn:
            if drop_table_first:
                conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`"))
                conn.commit()
                # 새로 생성
                first_chunk.to_sql(
                    name=table_name, con=conn, if_exists='replace',
                    index=False, chunksize=1000, method='multi'
                )
            else:
                # 존재 여부 확인 → 없으면 생성
                try:
                    conn.execute(text(f"SELECT 1 FROM `{table_name}` LIMIT 1"))
                except Exception:
                    first_chunk.to_sql(
                        name=table_name, con=conn, if_exists='replace',
                        index=False, chunksize=1000, method='multi'
                    )

            # UNIQUE 인덱스 보장 (중복 판단 기준)
            if dedup_keys:
                _ensure_unique_index(conn, table_name, dedup_keys)

        # 첫 청크는 다시 전체 루프에서 처리하도록 반환
        del first_chunk
        gc.collect()

        # 진행바
        pbar = tqdm(total=total_chunks, disable=not show_progress, desc=f"Saving {table_name}", ncols=100)
        newly_inserted = 0

        for i in range(total_chunks):
            start_idx = i * chunk_size
            end_idx = min((i + 1) * chunk_size, total_rows)
            chunk = df.iloc[start_idx:end_idx].copy()

            retry, max_retries = 0, 3
            while retry < max_retries:
                try:
                    with engine.connect() as conn:
                        if skip_existing and dedup_keys:
                            # 중복이면 무시(추가 적재 안함)
                            inserted = _insert_ignore_chunk(conn, table_name, chunk)
                            newly_inserted += inserted
                        else:
                            # 일반 append (중복 여부 신경 안 씀)
                            chunk.to_sql(
                                name=table_name, con=conn, if_exists='append',
                                index=False, chunksize=1000, method='multi'
                            )
                            newly_inserted += len(chunk)
                    break
                except Exception as e:
                    retry += 1
                    tqdm.write(f"   ⚠️  Chunk {i+1}/{total_chunks} failed (attempt {retry}/{max_retries}): {e}")
                    if retry < max_retries:
                        time.sleep(2)
                    else:
                        tqdm.write(f"   ❌ Chunk {i+1} failed after {max_retries} attempts")
                        raise
                finally:
                    del chunk
                    gc.collect()

            pbar.update(1)
            if i % 10 == 0 or i == total_chunks - 1:
                progress = (min(i + 1, total_chunks) / total_chunks) * 100
                tqdm.write(f"   📈 Progress: {newly_inserted:,} newly inserted, chunk {i+1}/{total_chunks} ({progress:.1f}%)")

            if i % 50 == 0:
                time.sleep(1)

        pbar.close()
        print(f"   ✅ Insert completed. Newly inserted rows: {newly_inserted:,}")
        if skip_existing:
            print("   ℹ️  Duplicates (by UNIQUE index) were ignored and NOT inserted.")
        return True

    except Exception as e:
        print(f"   ❌ Error saving {table_name}: {e}")
        return False


# =========================================
# 실행부
# =========================================
print("=" * 70)
print("SAVING FINANCIAL DATA TO MySQL DATABASE (LARGE DATA + PROGRESS + NO DUP INSERT)")
print("=" * 70)

try:
    # 연결 테스트
    with engine.connect() as conn:
        print("✓ Database connection successful")

    saved_tables = []

    # (예시) 재무 데이터 유니크 키
    dedup_keys = ['ticker', 'date', 'item']   # 필요 시 조정

    # 1) Income Statement
    if 'IS' in locals() and IS is not None and isinstance(IS, pd.DataFrame) and not IS.empty:
        print("\n1. Saving Income Statement data...")
        print(f"   - Data shape: {IS.shape}")
        print(f"   - Table name: US_IS_from_FMP")

        ok = save_large_dataframe_safely(
            IS, 'US_IS_from_FMP', engine,
            chunk_size=5000,
            show_progress=True,
            dedup_keys=dedup_keys,
            drop_table_first=False,    # 기존 데이터 유지
            skip_existing=True         # 과거 중복은 무시(추가 적재 금지)
        )
        if ok:
            saved_tables.append('US_IS_from_FMP')
            print("   ✅ Income Statement data saved successfully")
    else:
        print("\n1. Income Statement: No data to save (IS variable not found or empty)")

    # 2) Balance Sheet
    if 'BS' in locals() and BS is not None and isinstance(BS, pd.DataFrame) and not BS.empty:
        print("\n2. Saving Balance Sheet data...")
        print(f"   - Data shape: {BS.shape}")
        print(f"   - Table name: US_BS_from_FMP")

        ok = save_large_dataframe_safely(
            BS, 'US_BS_from_FMP', engine,
            chunk_size=5000,
            show_progress=True,
            dedup_keys=dedup_keys,
            drop_table_first=False,
            skip_existing=True
        )
        if ok:
            saved_tables.append('US_BS_from_FMP')
            print("   ✅ Balance Sheet data saved successfully")
    else:
        print("\n2. Balance Sheet: No data to save (BS variable not found or empty)")

    # 3) Cash Flow
    if 'CF' in locals() and CF is not None and isinstance(CF, pd.DataFrame) and not CF.empty:
        print("\n3. Saving Cash Flow data...")
        print(f"   - Data shape: {CF.shape}")
        print(f"   - Table name: US_CF_from_FMP")

        ok = save_large_dataframe_safely(
            CF, 'US_CF_from_FMP', engine,
            chunk_size=5000,
            show_progress=True,
            dedup_keys=dedup_keys,
            drop_table_first=False,
            skip_existing=True
        )
        if ok:
            saved_tables.append('US_CF_from_FMP')
            print("   ✅ Cash Flow data saved successfully")
    else:
        print("\n3. Cash Flow: No data to save (CF variable not found or empty)")

    # =========================================
    # 저장 검증
    # =========================================
    if saved_tables:
        print("\n" + "=" * 50)
        print("VERIFYING SAVED DATA")
        print("=" * 50)

        with engine.connect() as conn:
            for table_name in saved_tables:
                try:
                    # 행 수
                    row_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).fetchone()[0]
                    # 날짜 범위
                    min_date, max_date = conn.execute(text(f"SELECT MIN(date), MAX(date) FROM `{table_name}`")).fetchone()
                    # 유니크 티커 수
                    ticker_count = conn.execute(text(f"SELECT COUNT(DISTINCT ticker) FROM `{table_name}`")).fetchone()[0]

                    print(f"\n📊 {table_name}:")
                    print(f"   - Total records: {row_count:,}")
                    print(f"   - Unique tickers: {ticker_count:,}")
                    if min_date and max_date:
                        print(f"   - Date range: {min_date} to {max_date}")

                    # 샘플 데이터
                    sample = conn.execute(text(f"SELECT ticker, item, value FROM `{table_name}` LIMIT 3")).fetchall()
                    if sample:
                        print("   - Sample data:")
                        for i, (ticker, item, value) in enumerate(sample, 1):
                            print(f"     {i}. {ticker} | {item} | {value}")

                except Exception as verify_error:
                    print(f"❌ Error verifying {table_name}: {verify_error}")

    print("\n" + "=" * 70)
    print("DATABASE SAVE OPERATION COMPLETED!")
    print("=" * 70)

    if saved_tables:
        print(f"✅ Successfully handled {len(saved_tables)} tables: {saved_tables}")
    else:
        print("❌ No tables were saved")

except Exception as e:
    print("❌ Error occurred during database operations:")
    print(f"   Error type: {type(e).__name__}")
    print(f"   Error message: {str(e)}")

    # 연결 문제 진단
    try:
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
    except Exception as conn_error:
        print("\n💡 Connection issue detected:")
        print(f"   - Check if MySQL server is running")
        print(f"   - Verify host: {db_info['host']}")
        print(f"   - Verify port: {db_info['port']}")
        print(f"   - Verify database: {db_info['database']}")
        print(f"   - Verify credentials")

finally:
    if 'engine' in locals():
        engine.dispose()
        print("\n✓ Database connection closed")

gc.collect()
print("\n✓ Memory cleanup completed")


SAVING FINANCIAL DATA TO MySQL DATABASE (OPTIMIZED FOR LARGE DATA)
✓ Database connection successful

1. Saving Income Statement data...
   - Data shape: (20495265, 8)
   - Table name: US_IS_from_FMP
   📊 Processing 20,495,265 rows in chunks of 5,000


In [37]:
len(CF['ticker'].unique().tolist())

6047